# Sauti: build the fine-tuned phone model (Gemma 3n E2B -> .litertlm)

Run the cells in order (Shift+Enter, or Runtime -> Run all). About 45-60 minutes, mostly waiting.

**Runtime:** Runtime -> Change runtime type. Works on the **free v5e-1 TPU** runtime (51 GB RAM; the TPU itself is not used)
or on Colab Pro **CPU + High-RAM**. The free runtime has a daily allowance: start early in the day with a fresh allowance.

What it does: downloads Google's Gemma 3n E2B, merges our 85 MB Swahili/Sheng LoRA adapter, converts it to the phone
format (int4), adds Google's unchanged audio encoder, and saves `sauti-e2b-v1.litertlm` to Google Drive -> My Drive -> Sauti.

Restart-safe: the upload zip is kept in Drive after the first run, and cell 6 skips any step whose output already exists.

In [ ]:
# 1. Check the machine
import os, shutil
ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
disk = shutil.disk_usage('/content').free / 1e9
print(f"RAM {ram:.0f} GB, free disk {disk:.0f} GB, CPUs {os.cpu_count()}")
assert ram >= 45, "Need 45 GB+ RAM: free v5e-1 TPU runtime, or Colab Pro CPU + High-RAM"
assert disk >= 60, "Need 60 GB+ free disk"

In [ ]:
# 2. Connect Google Drive (approve the pop-up). The finished model is saved there.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Get sauti-cloud.zip: from Drive if a previous run saved it, otherwise upload it (Desktop > Sauti-cloud)
import os, shutil
saved = '/content/drive/MyDrive/Sauti/sauti-cloud.zip'
if not os.path.exists('/content/sauti-cloud.zip'):
    if os.path.exists(saved):
        shutil.copy(saved, '/content/sauti-cloud.zip'); print('copied from Drive')
    else:
        from google.colab import files
        files.upload()
        os.makedirs(os.path.dirname(saved), exist_ok=True); shutil.copy('/content/sauti-cloud.zip', saved); print('saved a copy to Drive')
!rm -rf /content/sauti && mkdir -p /content/sauti && unzip -q -o /content/sauti-cloud.zip -d /content/sauti && ls /content/sauti

In [ ]:
%%bash
# 4. Install tools in two environments (merge and convert need different library versions), ~5 min
set -e
pip -q install uv
[ -x /content/merge-env/bin/python ] || {
  uv venv -q /content/merge-env --python 3.12
  uv pip install -q --python /content/merge-env/bin/python torch==2.5.1 --index-url https://download.pytorch.org/whl/cpu
  uv pip install -q --python /content/merge-env/bin/python transformers==4.57.1 peft==0.19.1 accelerate==1.14.0 timm==1.0.27 pillow sentencepiece huggingface_hub==0.36.2 'numpy<2.3'
}
[ -x /content/conv-env/bin/litert-torch ] || {
  uv venv -q /content/conv-env --python 3.12
  uv pip install -q --python /content/conv-env/bin/python torch==2.14.1 torchvision==0.29.2 --index-url https://download.pytorch.org/whl/cpu
  uv pip install -q --python /content/conv-env/bin/python -r /content/sauti/requirements-convert.txt
}
/content/conv-env/bin/python -c "import litert_torch, jax; print('converter ready, jax', jax.__version__)"
/content/merge-env/bin/python -c "import peft, transformers; print('merger ready', transformers.__version__, peft.__version__)"

In [ ]:
# 5. Sign in to Hugging Face: open the link it shows, enter the code, approve.
#    (Your account must have accepted Gemma's licence. No token to paste.)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
%%bash
# 6. Download, merge, convert to phone format, add audio, save to Drive (~35-50 min)
set -eo pipefail
trap 'echo FAILED; tail -15 /content/download.log /content/merge.log 2>/dev/null' ERR
export JAX_PLATFORMS=cpu
cd /content
M=/content/merge-env/bin/python
C=/content/conv-env/bin
FLAGS="--task=text_generation --quantization_recipe=dynamic_wi4_afp32 --cache_length=4096 --use_jinja_template=True --bundle_litert_lm=True --externalize_embedder=True --model_dtype=float32 --experimental_lightweight_conversion=True"

echo "$(date +%H:%M) downloading Google's models"
$M -c "from huggingface_hub import snapshot_download as s, hf_hub_download as h; s('google/gemma-3n-E2B-it', local_dir='/content/gemma-3n-E2B-it'); h('google/gemma-3n-E2B-it-litert-lm', 'gemma-3n-E2B-it-int4.litertlm', local_dir='/content/google')" > /content/download.log 2>&1

if [ ! -f /content/merged-e2b-v1/config.json ]; then
  echo "$(date +%H:%M) merging our Swahili/Sheng adapter"
  $M /content/sauti/merge_lora.py --model /content/gemma-3n-E2B-it --adapter /content/sauti/adapter --out /content/merged-e2b-v1 > /content/merge.log 2>&1
fi

if ! ls /content/litert-out/*.litertlm > /dev/null 2>&1; then
  echo "$(date +%H:%M) converting to phone format, low-memory mode (longest step)"
  rm -rf /content/litert-out
  if true; then  # straight to low-memory mode: the full conversion runs out of memory on 47 GB
    # Usually out of memory at the per-layer embedder. That table is unchanged by our LoRA,
    # so convert without it and take Google's copy in the next step.
    rm -rf /content/litert-out
    $C/python /content/sauti/export_lowmem.py export_hf /content/merged-e2b-v1 /content/litert-out $FLAGS > /content/export-lowmem.log 2>&1 || { grep -v '^\[ERROR\]' /content/export-lowmem.log | tail -30; exit 1; }
  fi
fi
ls -lh /content/litert-out

echo "$(date +%H:%M) adding Google's audio parts"
OURS=$(ls /content/litert-out/*.litertlm | head -1)
rm -rf /content/parts-ours /content/parts-google
$C/litert-lm-peek --litertlm_file "$OURS" --dump_files_dir /content/parts-ours > /dev/null
$C/litert-lm-peek --litertlm_file /content/google/gemma-3n-E2B-it-int4.litertlm --dump_files_dir /content/parts-google > /dev/null
$C/python /content/sauti/splice_audio.py --base-parts /content/parts-google --ours-parts /content/parts-ours --out /content/sauti-e2b-v1.litertlm --builder $C/litert-lm-builder
$C/litert-lm-peek --litertlm_file /content/sauti-e2b-v1.litertlm | grep -E 'model_type' || true

echo "$(date +%H:%M) saving to Google Drive"
mkdir -p /content/drive/MyDrive/Sauti && cp /content/sauti-e2b-v1.litertlm /content/drive/MyDrive/Sauti/ && ls -lh /content/drive/MyDrive/Sauti
echo "$(date +%H:%M) DONE: Google Drive > My Drive > Sauti > sauti-e2b-v1.litertlm"